# DermaSense: Comprehensive Exploratory Data Analysis & Feature Insights
### Personalized, Explainable AI Skincare Product Recommendation System

This notebook provides visual analysis of the cleaned and standardized **DermaSense** datasets:
1. Master Product Catalog (`data/processed/master_product_features.parquet`)
2. Unified Customer Reviews (`data/processed/unified_reviews.parquet`)

---

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Paths
PROCESSED_DIR = "../data/processed"
prods_path = os.path.join(PROCESSED_DIR, "master_product_features.parquet")
revs_path = os.path.join(PROCESSED_DIR, "unified_reviews.parquet")

df_prods = pd.read_parquet(prods_path)
df_revs = pd.read_parquet(revs_path)

print(f"Master Products: {df_prods.shape[0]:,} rows x {df_prods.shape[1]} columns")
print(f"Unified Reviews: {df_revs.shape[0]:,} rows x {df_revs.shape[1]} columns")

## 1. Product Category Distribution across Platforms

In [ ]:
plt.figure(figsize=(12, 6))
cat_counts = df_prods['category'].value_counts().head(10)
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette="viridis")
plt.title("Top 10 Skincare Categories in DermaSense", fontsize=14, fontweight='bold')
plt.xlabel("Number of Products")
plt.ylabel("Category")
for i, v in enumerate(cat_counts.values):
    plt.text(v + 30, i, f"{v:,}", va='center', fontweight='semibold')
plt.tight_layout()
plt.show()

## 2. Price Distribution by Product Category (USD & INR)

In [ ]:
top_cats = df_prods['category'].value_counts().head(6).index
df_price_filtered = df_prods[df_prods['category'].isin(top_cats) & (df_prods['price_usd'] > 0) & (df_prods['price_usd'] <= 150)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_price_filtered, x='category', y='price_usd', palette="Set2")
plt.title("Price Distribution (USD) across Top Skincare Categories (< $150)", fontsize=14, fontweight='bold')
plt.xlabel("Category")
plt.ylabel("Price (USD $)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 3. Targeted Skincare Benefits Coverage from Ingredient Ontology

In [ ]:
benefit_cols = [c for c in df_prods.columns if c.startswith('benefit_')]
benefit_labels = [c.replace('benefit_', '').replace('___', ' & ').replace('_', ' ').title() for c in benefit_cols]
benefit_totals = [df_prods[c].sum() for c in benefit_cols]

df_benefits = pd.DataFrame({'Benefit': benefit_labels, 'Count': benefit_totals}).sort_values('Count', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_benefits, x='Count', y='Benefit', palette="coolwarm")
plt.title("Skincare Benefits Identified from Ingredients & Descriptions", fontsize=14, fontweight='bold')
plt.xlabel("Number of Matching Products")
for i, v in enumerate(df_benefits['Count']):
    pct = (v / len(df_prods)) * 100
    plt.text(v + 30, i, f"{v:,} ({pct:.1f}%)", va='center')
plt.tight_layout()
plt.show()

## 4. User Skin Type Distribution in Reviews

In [ ]:
skin_counts = df_revs['user_skin_type'].value_counts()

plt.figure(figsize=(10, 6))
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#bab0ac']
plt.pie(skin_counts.values, labels=skin_counts.index.str.title(), autopct='%1.1f%%', startangle=140, colors=colors)
plt.title("User Skin Type Annotations in DermaSense Reviews (1.20M Reviews)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Skin-Type Specific Product Satisfaction Comparison
How do users with different skin types rate the same product?

In [ ]:
# Find products with substantial reviews across all skin types
mask_rich = (
    (df_prods['oily_skin_review_count'] >= 50) &
    (df_prods['dry_skin_review_count'] >= 50) &
    (df_prods['combination_skin_review_count'] >= 100)
)
df_rich = df_prods[mask_rich].copy()

print(f"Found {len(df_rich)} products with 50+ reviews across multiple skin types.")

df_rich['oily_vs_dry_diff'] = df_rich['oily_skin_avg_rating'] - df_rich['dry_skin_avg_rating']
top_oily_favs = df_rich.sort_values('oily_vs_dry_diff', ascending=False)[['product_name', 'brand', 'category', 'oily_skin_avg_rating', 'dry_skin_avg_rating', 'oily_vs_dry_diff']].head(5)
top_dry_favs = df_rich.sort_values('oily_vs_dry_diff', ascending=True)[['product_name', 'brand', 'category', 'oily_skin_avg_rating', 'dry_skin_avg_rating', 'oily_vs_dry_diff']].head(5)

print("\n--- Top Products Favored Significantly More by Oily Skin than Dry Skin ---")
display(top_oily_favs)

print("\n--- Top Products Favored Significantly More by Dry Skin than Oily Skin ---")
display(top_dry_favs)

## 6. Review Text Length and Helpfulness Metrics

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_prods['avg_review_char_length'].dropna(), bins=40, color='teal', kde=True)
plt.title("Average Review Length per Product (Characters)", fontweight='bold')
plt.xlabel("Avg Characters")

plt.subplot(1, 2, 2)
sns.scatterplot(data=df_prods[df_prods['review_count_actual'] > 0], x='review_count_actual', y='avg_rating_calculated', alpha=0.5, color='coral')
plt.xscale('log')
plt.title("Product Rating vs Review Volume (Log Scale)", fontweight='bold')
plt.xlabel("Review Count (Log Scale)")
plt.ylabel("Average Rating")

plt.tight_layout()
plt.show()